In [1]:
!pip install transformers[torch] pandas openpyxl scikit-learn

In [7]:
import pandas as pd

df = pd.read_csv('labeled_data.csv')
print("5 dòng dữ liệu đầu tiên:")
print(df.head())
print("Thống kê số lượng bài báo mỗi chuyên mục:")
print(df['category'].value_counts())

5 dòng dữ liệu đầu tiên:
                                                 url  \
0  https://vnexpress.net/loat-loi-tren-ios-26-494...   
1  https://tuoitre.vn/tiktok-thoat-hiem-tai-my-20...   
2  https://dantri.com.vn/kinh-doanh/chenh-lech-gi...   
3  https://vnexpress.net/doanh-nghiep-nao-dang-mu...   
4  https://tuoitre.vn/60-kol-quan-tri-vien-mang-x...   

                                               title  \
0                               Loạt lỗi trên iOS 26   
1                           TikTok thoát hiểm tại Mỹ   
2  Chênh lệch giá mua bán vàng nhẫn vẫn ở 3,6 tri...   
3        Doanh nghiệp nào đang muốn lập sàn tiền số?   
4  60 KOL, quản trị viên mạng xã hội cam kết khôn...   

                                                text   category  
0  Nhiều người dùng nâng cấp iOS 26 cho biết iPho...  cong-nghe  
1  Thỏa thuận khung về TikTok không chỉ cứu ứng d...  cong-nghe  
2  (Dân trí) - Trên thế giới, giá vàng tăng nhẹ l...    kinh-te  
3  Nhiều ngân hàng và công ty chứng k

In [8]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

MODEL_NAME = "vinai/phobert-base-v2"

try:
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=8)
    print("✅ Tải PhoBERT model và tokenizer thành công!")
except Exception as e:
    print(f"❌ Lỗi khi tải model: {e}")

/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(
Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at vinai/phobert-base-v2 and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


✅ Tải PhoBERT model và tokenizer thành công!


In [9]:
# Lấy danh sách chuyên mục duy nhất
unique_categories = df['category'].unique().tolist()

# Bản đồ chuyển đổi
label2id = {label: i for i, label in enumerate(unique_categories)}
id2label = {i: label for i, label in enumerate(unique_categories)}

# Tạo cột nhãn số
df['label'] = df['category'].map(label2id)
print("label2id:", label2id)

label2id: {'cong-nghe': 0, 'kinh-te': 1, 'giao-duc': 2, 'giai-tri': 3, 'suc-khoe': 4, 'the-thao': 5, 'phap-luat': 6, 'xa-hoi': 7}


In [10]:
from sklearn.model_selection import train_test_split

train_texts, test_texts, train_labels, test_labels = train_test_split(
    df['text'].tolist(),
    df['label'].tolist(),
    test_size=0.2,
    random_state=42,
    stratify=df['label']
)

In [11]:
train_encodings = tokenizer(train_texts, truncation=True, padding=True, max_length=256)
test_encodings  = tokenizer(test_texts,  truncation=True, padding=True, max_length=256)

In [12]:
import torch

class NewsDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels
    def __getitem__(self, idx):
        item = {k: torch.tensor(v[idx]) for k, v in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item
    def __len__(self):
        return len(self.labels)

train_dataset = NewsDataset(train_encodings, train_labels)
test_dataset  = NewsDataset(test_encodings,  test_labels)

In [7]:
from transformers import Trainer, TrainingArguments
import numpy as np
from sklearn.metrics import accuracy_score, f1_score

def compute_metrics(pred):
    labels = pred.label_ids
    preds = np.argmax(pred.predictions, axis=1)
    acc = accuracy_score(labels, preds)
    f1  = f1_score(labels, preds, average='weighted')
    return {"accuracy": acc, "f1": f1}

training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=4,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    logging_dir='./logs',
    logging_steps=10,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
)

TypeError: TrainingArguments.__init__() got an unexpected keyword argument 'evaluation_strategy'

In [10]:
!pip install -U transformers

In [8]:
!pip uninstall -y transformers tokenizers accelerate

Found existing installation: transformers 4.57.0
Uninstalling transformers-4.57.0:
  Successfully uninstalled transformers-4.57.0
Found existing installation: tokenizers 0.22.1
Uninstalling tokenizers-0.22.1:
  Successfully uninstalled tokenizers-0.22.1
Found existing installation: accelerate 1.10.1
Uninstalling accelerate-1.10.1:
  Successfully uninstalled accelerate-1.10.1


In [9]:
!pip install -U "transformers[torch]==4.44.0" accelerate datasets

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 95.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 374.9/374.9 kB 32.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.3/506.3 kB 42.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.8/42.8 MB 22.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 110.8 MB/s eta 0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
pylibcudf-cu12 25.6.0 requires

In [1]:
import transformers
print("transformers version:", transformers.__version__)

The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.


0it [00:00, ?it/s]

transformers version: 4.44.0


In [13]:
from transformers import Trainer, TrainingArguments
import numpy as np
from sklearn.metrics import accuracy_score, f1_score

def compute_metrics(pred):
    labels = pred.label_ids
    preds = np.argmax(pred.predictions, axis=1)
    acc = accuracy_score(labels, preds)
    f1  = f1_score(labels, preds, average='weighted')
    return {"accuracy": acc, "f1": f1}

training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=4,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    logging_dir='./logs',
    logging_steps=10,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
)

/usr/local/lib/python3.12/dist-packages/transformers/training_args.py:1525: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


In [15]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics,
)

trainer.train()

<IPython.core.display.Javascript object>

wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize?ref=models
wandb: Paste an API key from your profile and hit enter:

 ··········


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: huyentrangn337 (huyentrangn337-ht) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,2.017000,1.621239,0.771930,0.720517
2,1.294300,1.094659,0.842105,0.820141
3,0.896600,0.894816,0.842105,0.820141
4,0.728000,0.845242,0.842105,0.829569


TrainOutput(global_step=112, training_loss=1.260461827473981, metrics={'train_runtime': 194.2354, 'train_samples_per_second': 4.613, 'train_steps_per_second': 0.577, 'total_flos': 117880102846464.0, 'train_loss': 1.260461827473981, 'epoch': 4.0})

In [16]:
trainer.save_model("my-finetuned-phobert")
tokenizer.save_pretrained("my-finetuned-phobert")

('my-finetuned-phobert/tokenizer_config.json',
 'my-finetuned-phobert/special_tokens_map.json',
 'my-finetuned-phobert/vocab.txt',
 'my-finetuned-phobert/bpe.codes',
 'my-finetuned-phobert/added_tokens.json')

In [17]:
metrics = trainer.evaluate(test_dataset)
print("🎯 Độ chính xác (Accuracy) trên tập test:", metrics["eval_accuracy"])
print("📊 F1-score:", metrics["eval_f1"])

🎯 Độ chính xác (Accuracy) trên tập test: 0.8421052631578947
📊 F1-score: 0.8295689690426532


In [18]:
from google.colab import files
!zip -r my-finetuned-phobert.zip my-finetuned-phobert
files.download("my-finetuned-phobert.zip")

  adding: my-finetuned-phobert/ (stored 0%)
  adding: my-finetuned-phobert/tokenizer_config.json (deflated 77%)
  adding: my-finetuned-phobert/bpe.codes (deflated 59%)
  adding: my-finetuned-phobert/vocab.txt (deflated 55%)
  adding: my-finetuned-phobert/special_tokens_map.json (deflated 57%)
  adding: my-finetuned-phobert/added_tokens.json (stored 0%)
  adding: my-finetuned-phobert/training_args.bin (deflated 53%)
  adding: my-finetuned-phobert/config.json (deflated 56%)
  adding: my-finetuned-phobert/model.safetensors (deflated 7%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [19]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [20]:
import os

# Đường dẫn lưu trên Drive
base_path = "/content/drive/MyDrive/ai_notebooks"

# Tạo các thư mục con
os.makedirs(f"{base_path}/models", exist_ok=True)
os.makedirs(f"{base_path}/logs", exist_ok=True)
os.makedirs(f"{base_path}/notebooks", exist_ok=True)

# Tạo file README.md mẫu
readme_content = """# 📘 ai_notebooks

Thư mục này lưu trữ kết quả huấn luyện mô hình AI (PhoBERT fine-tuning).

## Cấu trúc thư mục
- **models/** : chứa các mô hình đã huấn luyện (.zip hoặc folder)
- **logs/** : chứa file log hoặc csv ghi kết quả
- **notebooks/** : chứa các notebook (.ipynb)
- **README.md** : mô tả thông tin huấn luyện, thông số, kết quả và nhận xét

## Ghi chú:
Mỗi lần huấn luyện, ghi lại:
- Thời gian train
- Thông số (epochs, batch_size, lr, max_length)
- Kết quả (Accuracy, F1)
- Nhận xét, lỗi gặp phải, cải tiến
"""

with open(f"{base_path}/README.md", "w", encoding="utf-8") as f:
    f.write(readme_content)

print("✅ Đã tạo thư mục 'ai_notebooks/' trong Google Drive thành công!")
print(f"📂 Đường dẫn: {base_path}")


✅ Đã tạo thư mục 'ai_notebooks/' trong Google Drive thành công!
📂 Đường dẫn: /content/drive/MyDrive/ai_notebooks


In [21]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics,
)

trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.672600,0.714566,0.842105,0.830197
2,0.299900,0.599511,0.859649,0.856347
3,0.143300,0.574194,0.877193,0.875240
4,0.111600,0.657989,0.859649,0.853262


TrainOutput(global_step=112, training_loss=0.3086407887084143, metrics={'train_runtime': 107.2841, 'train_samples_per_second': 8.352, 'train_steps_per_second': 1.044, 'total_flos': 117880102846464.0, 'train_loss': 0.3086407887084143, 'epoch': 4.0})